# T05 – Diversity & mood over the years
*MIR & ML for Audio · TU Berlin · WiSe 2026/27*

| | |
|---|---|
| **Name** | Hallvard Jensen |
| **Student ID** | 0549158 |
| **Corpus (genre)** | Nu Disco/Disco |

**Submission:** work in this notebook – code, figures and text answers all stay here. Before uploading: **Restart Kernel & Run All**, check that every output is shown, then upload this `.ipynb` file.

**Documentation:** document your work directly in the notebook – comments in your code cells and, where helpful, short markdown cells.

### Sources & AI tools
- **Sources** (literature, data, code):
- **LLM coding assistants** (tool/model – how and what for):

## Tasks
1. Examine how label, artist and gender diversity in your EDMC genre have developed over the years!
   - *Work with the complete chart metadata (CSV file).*
   - Enrich the metadata with gender information in a meaningful way
   - Provide an overview figure on the completeness of the information on labels, artists and gender
   - Create figures showing the development of the three diversities over the years
   - Summarise the results in one sentence a layperson can understand
2. Check whether there have been clear breaks in the emotional mood of the Top 100 tracks of your EDMC genre over the years.
   - Use the mood classifiers of Essentia
   - Visualise the development of mood over the years
   - Use a suitable self-similarity metric to compare the years
   - Create a Foote novelty curve plot to identify possible breaks
   - Summarise the results in one sentence a layperson can understand

---
## Setup

Expensive steps (feature extraction, separation, model training): save the results to `OUTPUT_DIR` (e.g. as CSV) and load them on the next run instead of recomputing – *Run All* stays fast. Results of earlier sessions are in `output/T02/`, `output/T03/`, …

In [2]:
# import standard packages you'll need

import sys
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(sys.executable)

# ─ add your own additional imports here:
from collections.abc import Callable

import discogs_client
import musicbrainzngs
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.utils import default_process
import tqdm

/home/hallvard/miniforge3/envs/mirml/bin/python


In [3]:
# ── CONFIG ───────────────────────────────────────────────────────────────
CORPUS_DIR = Path("data") / "nu-disco-disco"   # ← name of your corpus folder
# ─────────────────────────────────────────────────────────────────────────
SESSION = "T05"
PREVIEW_DIR = CORPUS_DIR / "previews"
OUTPUT_DIR = Path("output") / SESSION      # results of this session
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert CORPUS_DIR.is_dir(), f"Not found: {CORPUS_DIR.resolve()} → check CORPUS_DIR"
CHART_CSV = next(CORPUS_DIR.glob("*.csv"))
print(CHART_CSV)

def output_filename(task: str, filename: str) -> Path:
    return OUTPUT_DIR / f"t05_{task}_nu_disco_disco_0549158_{filename}"



data/nu-disco-disco/Nu_Disco_Disco.csv


---
## Task 1
**Examine how label, artist and gender diversity in your EDMC genre have developed over the years!**

*Work with the complete chart metadata (CSV file).*

### 1.1 · Enrich the metadata with gender information in a meaningful way

In [ ]:
# 1.1
musicbrainzngs.set_useragent("MIRML-T05", "0.1", "hjjensen@campus.tu-berlin.de")

from pprint import pprint


def similarity(a: str, b: str) -> float:
    return JaroWinkler.normalized_similarity(a, b, processor=default_process)


def get_artist_by_isrc(isrc: str) -> list[tuple[str, str]]:
    if pd.isna(isrc) or not isrc.strip():
        return []
    try:
        r = musicbrainzngs.get_recordings_by_isrc(isrc, includes=["artists"])
    except musicbrainzngs.ResponseError as ex:
        return []
    rl = r["isrc"]["recording-list"]
    pprint(rl)
    if rl:
        artist_credit = rl[0].get("artist-credit", [])
    else:
        artist_credit = []
    return [
        (credit["artist"]["id"], credit["artist"]["name"])
        for credit in artist_credit
        if isinstance(credit, dict)
    ]


def get_artist_from_name(name: str) -> list:
    for hit in musicbrainzngs.search_artists(artist=name, limit=5)[
        "artist-list"
    ]:
        score = int(hit.get("ext:score", 0))
        if score >= 90 and similarity(name, hit["name"]) >= 0.9:
            return hit["id"], hit["name"]
    return None, name


def dget(d: dict, key: str):
    return d[key] if key in d else None


def get_artist_info(mb_id: int):

    if not mb_id:
        cache[mb_id] = {"mb_type": None, "mb_gender": None, "mb_country": None}

    if mb_id not in cache:
        artist = musicbrainzngs.get_artist_by_id(mb_id)["artist"]
        cache[mb_id] = {
            "mb_type": dget(artist, "type"),
            "mb_gender": dget(artist, "gender"),
            "mb_country": dget(artist, "country"),
        }

    return cache[mb_id]


def get_label():
    musicbrainzngs.search_labels()


def get_label_info(label_id: int):
    res = musicbrainzngs.get_label_by_id(label_id)
    print(res)


def mb_lookup(row: pd.Series):
    isrc = row["isrc"]
    assert type(isrc) == str

    route = "isrc"
    artists = get_artist_by_isrc(isrc)
    if not artists:
        route = "name"
        artists = [
            get_artist_from_name(name.strip())
            for name in row["artist"].split(",")
        ]

    return [
        {
            "route": route if mb_id else None,
            "mb_artist_id": mb_id,
            "mb_artist": name,
            **(get_artist_info(mb_id)),
        }
        for mb_id, name in artists
    ]


def enrich(items: pd.DataFrame, lookup_fn: Callable, cache_csv: Path, key: str):
    if cache_csv.exists():
        done = set(pd.read_csv(cache_csv))
    else:
        done = set()

    for _, row in tqdm.tqdm(items[~items[key].isin(done)].iterrows()):
        try:
            res = lookup_fn(row)
        except Exception as ex:
            print(f"Failed to enrich row with key {row[key]}: ex")
            continue

        if res:
            df = pd.DataFrame([{key: row[key], **a} for a in res])
            df.to_csv(
                cache_csv, mode="a", header=not cache_csv.exists(), index=False
            )
    return pd.read_csv(cache_csv)


cache = {}
charts = pd.read_csv(CHART_CSV)
tracks = (
    charts[charts["beatport_track_id"] > 0]
    .copy()
    .drop_duplicates("beatport_track_id")
    .dropna(subset=["artist", "title"])
    .loc[:, ["beatport_track_id", "artist", "title", "isrc"]]
    .astype({"beatport_track_id": int})
    .reset_index(drop=True)
)

row = tracks.iloc[100]
mb_lookup(row)

# mb = enrich(
#     tracks,
#     mb_lookup,
#     output_filename("1.1", "mb_artists.csv"),
#     "beatport_track_id",
# )
# mb

[{'artist-credit': [{'artist': {'country': 'FR',
                                'id': 'd6285d95-8ee1-4de2-8ceb-ff90098ab1d1',
                                'name': 'Yuksek',
                                'sort-name': 'Yuksek',
                                'type': 'Person'}},
                    ' feat. ',
                    {'artist': {'disambiguation': 'Electronic Music Singer',
                                'id': 'a5b7cca0-2cea-4337-a9cb-2f9d44ad7d3b',
                                'name': 'Queen Rose',
                                'sort-name': 'Queen Rose',
                                'type': 'Person'}}],
  'artist-credit-phrase': 'Yuksek feat. Queen Rose',
  'disambiguation': 'Explicit',
  'id': '174a8554-6210-497f-857b-c8e6aa82be06',
  'length': '205413',
  'title': 'G.F.Y.'},
 {'artist-credit': [{'artist': {'country': 'FR',
                                'id': 'd6285d95-8ee1-4de2-8ceb-ff90098ab1d1',
                                'name': 'Yuksek',
          

[{'route': 'isrc',
  'mb_artist_id': 'd6285d95-8ee1-4de2-8ceb-ff90098ab1d1',
  'mb_artist': 'Yuksek',
  'mb_type': 'Person',
  'mb_gender': 'Male',
  'mb_country': 'FR'},
 {'route': 'isrc',
  'mb_artist_id': 'a5b7cca0-2cea-4337-a9cb-2f9d44ad7d3b',
  'mb_artist': 'Queen Rose',
  'mb_type': 'Person',
  'mb_gender': 'Female',
  'mb_country': None}]

### 1.2 · Provide an overview figure on the completeness of the information on labels, artists and gender

In [ ]:
# 1.2


### 1.3 · Create figures showing the development of the three diversities over the years

In [ ]:
# 1.3


### 1.4 · Summarise the results in one sentence a layperson can understand

#### Answer 1.4
*Your answer*

---
## Task 2
**Check whether there have been clear breaks in the emotional mood of the Top 100 tracks of your EDMC genre over the years.**

### 2.1 · Use the mood classifiers of Essentia

In [ ]:
# 2.1


### 2.2 · Visualise the development of mood over the years

In [ ]:
# 2.2


### 2.3 · Use a suitable self-similarity metric to compare the years

In [ ]:
# 2.3


### 2.4 · Create a Foote novelty curve plot to identify possible breaks

In [ ]:
# 2.4


### 2.5 · Summarise the results in one sentence a layperson can understand

#### Answer 2.5
*Your answer*

---
## Before you upload
- [ ] Name, student ID and genre filled in
- [ ] **Restart Kernel & Run All** runs without errors, all outputs visible
- [ ] Figures labelled (axes, units, title); audio examples embedded where asked